In [2]:
from datetime import datetime

PROJECT_ID = "neurodocsdomain"
LOCATION = "us-west1"
UID = datetime.now().strftime("%m%d%H%M")

In [4]:
BUCKET_URI = f"gs://{PROJECT_ID}-vs-quickstart-{UID}"

In [6]:
! gcloud storage buckets create $BUCKET_URI --location=$LOCATION --project=$PROJECT_ID
! gcloud storage cp "gs://github-repo/data/vs-quickstart/product-embs.json" $BUCKET_URI

Creating gs://neurodocsdomain-vs-quickstart-01211106/...


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update

Copying gs://github-repo/data/vs-quickstart/product-embs.json to gs://neurodocsdomain-vs-quickstart-01211106/product-embs.json
  Completed files 1/1 | 79.3MiB/79.3MiB                                        


In [7]:
! gcloud storage cp "gs://github-repo/data/vs-quickstart/product-embs.json" . # for query tests

Copying gs://github-repo/data/vs-quickstart/product-embs.json to file://./product-embs.json
  Completed files 1/1 | 79.3MiB/79.3MiB | 12.3MiB/s                            

Average throughput: 13.1MiB/s


In [8]:
# init the aiplatform package
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=LOCATION)

In [11]:
from google.cloud.aiplatform.matching_engine import matching_engine_index_config

# create Index
my_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name = f"vs-quickstart-index-{UID}",
    contents_delta_uri = BUCKET_URI,
    dimensions = 768,
    approximate_neighbors_count = 100,
    distance_measure_type=matching_engine_index_config.DistanceMeasureType.SQUARED_L2_DISTANCE,
    leaf_node_embedding_count=100,
    leaf_nodes_to_search_percent=50,
    description="my description",
    labels={ "label_name": "label_value" },
)

Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/520885151616/locations/us-west1/indexes/1587369335066722304/operations/5706965900825985024
MatchingEngineIndex created. Resource name: projects/520885151616/locations/us-west1/indexes/1587369335066722304
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/520885151616/locations/us-west1/indexes/1587369335066722304')


In [12]:
## create `IndexEndpoint`
my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name = f"vs-quickstart-index-endpoint-{UID}",
    public_endpoint_enabled = True
)

Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808/operations/5673751853574127616
MatchingEngineIndexEndpoint created. Resource name: projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808')


In [13]:
DEPLOYED_INDEX_ID = f"vs_quickstart_deployed_{UID}"

In [14]:
# deploy the Index to the Index Endpoint
my_index_endpoint.deploy_index(
    index = my_index, deployed_index_id = DEPLOYED_INDEX_ID
)

Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808
Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808/operations/7948069665396228096
MatchingEngineIndexEndpoint index_endpoint Deployed index. Resource name: projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808


resource name: projects/520885151616/locations/us-west1/indexEndpoints/2114479607469047808

In [15]:
import json

# build dicts for product names and embs
product_names = {}
product_embs = {}
with open('product-embs.json') as f:
    for l in f.readlines():
        p = json.loads(l)
        id = p['id']
        product_names[id] = p['name']
        product_embs[id] = p['embedding']

In [19]:
print(product_names["6523"])

cloudveil women's excursion short


In [17]:
query_emb = product_embs['6523']
print(query_emb)

[-0.015140533447265625, 0.029022620990872383, 0.043999187648296356, 0.0008045680006034672, 0.02479265257716179, -0.058345310389995575, 0.010426630266010761, 0.023504989221692085, -0.03466186299920082, -0.00134370313026011, 0.007397875655442476, -0.01431096438318491, 0.024990102276206017, 0.06665688753128052, 0.023334601894021034, -0.005286165047436953, -0.06492510437965393, -0.0345313623547554, 0.060259561985731125, 0.010223621502518654, -0.09199754148721695, 0.01886577345430851, 0.03483972325921059, -0.027113549411296844, -0.03256196156144142, -0.07872982323169708, 0.037879571318626404, -0.009713241830468178, -0.03232517093420029, -0.07063174992799759, 0.0024606185033917427, -0.015956062823534012, -0.003946097567677498, 0.021167505532503128, -0.008327499032020569, 0.055032506585121155, 0.019084438681602478, 0.0015176940942183137, 0.00926684495061636, 0.06493163108825684, 0.0036904136650264263, 0.02693367190659046, 0.04891353100538254, -0.001483380445279181, -0.0366176962852478, -0.013

In [18]:
# run query
response = my_index_endpoint.find_neighbors(
    deployed_index_id = DEPLOYED_INDEX_ID,
    queries = [query_emb],
    num_neighbors = 10
)

# show the results
for idx, neighbor in enumerate(response[0]):
    print(f"{neighbor.distance:.2f} {product_names[neighbor.id]}")

0.00 cloudveil women's excursion short
0.36 quiksilver womens cruiser short
0.39 xcvi women's alisal short
0.40 cloudveil men's kahuna short
0.44 ibex women's gozo short
0.44 sanctuary clothing women's coquette short
0.45 sunner women's collins printed short
0.46 hurley lowrider cargo 2.5 short - women's
0.46 stitch's women's fox knee length short
0.47 sanctuary clothing women's passenger skirt
